# XGBoost Model for Delinquency Prediction

Uses the **top N selected features** from the comprehensive feature selection pipeline.

- Train / Validation / Test split (60/20/20)
- XGBoost binary classifier with early stopping
- Handles class imbalance with `scale_pos_weight`
- No feature scaling required for tree-based models
- Records training time, scoring time, ROC-AUC, and Average Precision

In [10]:
import sys
import pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    precision_recall_curve, average_precision_score, roc_curve,
)
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

from scripts.model_data import load_and_split

## 1. Load Data & Selected Features

In [11]:
# ── Configuration ────────────────────────────────────────────────
N_FEATURES = 50  # ← change this to use a different number of top-ranked features
FEATURES_FILE = "filtered_features_consensus_ordered.csv"

# Load features and perform 60/20/20 stratified split
X_train, X_val, X_test, y_train, y_val, y_test, available_features, features_df = \
    load_and_split(n_features=N_FEATURES, features_filename=FEATURES_FILE)

print(f'Features used:  {len(available_features)}')
print(f'Train:          {X_train.shape}  |  positives: {y_train.sum()} ({100*y_train.mean():.1f}%)')
print(f'Validation:     {X_val.shape}  |  positives: {y_val.sum()} ({100*y_val.mean():.1f}%)')
print(f'Test:           {X_test.shape}  |  positives: {y_test.sum()} ({100*y_test.mean():.1f}%)')

Features  : 50 (top-50)
Samples   : 9277  |  DQ rate: 8.70%
Train     : 5566  (8.70% positive)
Val       : 1855  (8.68% positive)
Test      : 1856  (8.73% positive)
Features used:  50
Train:          (5566, 50)  |  positives: 484 (8.7%)
Validation:     (1855, 50)  |  positives: 161 (8.7%)
Test:           (1856, 50)  |  positives: 162 (8.7%)


## 2. Prepare DMatrix Objects

XGBoost uses its own `DMatrix` format. No scaling is needed for tree-based models.

In [12]:
# Compute scale_pos_weight to handle class imbalance
n_neg = int(np.sum(y_train == 0))
n_pos = int(np.sum(y_train == 1))
scale_pos_weight = n_neg / n_pos
print(f'scale_pos_weight: {scale_pos_weight:.2f}  (neg={n_neg}, pos={n_pos})')

# Build DMatrix objects (XGBoost native format)
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=list(available_features))
dval   = xgb.DMatrix(X_val,   label=y_val,   feature_names=list(available_features))
dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=list(available_features))

print('\nDMatrix objects created successfully.')

scale_pos_weight: 10.50  (neg=5082, pos=484)

DMatrix objects created successfully.


## 3. Hyperparameter Tuning with Optuna

Use **Optuna** (TPE sampler) to minimise overfitting and maximise validation AUC.  
The search targets the key regularisation knobs: tree depth, leaf weight, gamma, L1/L2, and sampling rates.

In [15]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS      = 80    # increase for a more thorough search
EARLY_STOPPING = 30
NUM_BOOST_ROUND = 1000

def objective(trial):
    p = {
        'objective':         'binary:logistic',
        'eval_metric':       'auc',
        'scale_pos_weight':  scale_pos_weight,
        'seed':              42,
        'verbosity':         0,
        # --- tree complexity ---
        'max_depth':         trial.suggest_int('max_depth', 2, 6),
        'min_child_weight':  trial.suggest_int('min_child_weight', 5, 50),
        'gamma':             trial.suggest_float('gamma', 0.0, 5.0),
        # --- regularisation ---
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 10.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1.0, 20.0),
        # --- stochastic sampling ---
        'eta':               trial.suggest_float('eta', 0.01, 0.1, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 0.9)
    }

    pruning_cb = optuna.integration.XGBoostPruningCallback(trial, 'val-auc')

    bst = xgb.train(
        p,
        dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        evals=[(dtrain, 'train'), (dval, 'val')],
        early_stopping_rounds=EARLY_STOPPING,
        verbose_eval=False,
        callbacks=[pruning_cb],
    )
    return bst.best_score   # maximise val AUC

sampler = optuna.samplers.TPESampler(seed=42)
pruner  = optuna.pruners.MedianPruner(n_warmup_steps=10)
study   = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)

print(f'Running Optuna search ({N_TRIALS} trials) …')
t_tune = time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f'Search finished in {time.time()-t_tune:.1f}s')
print(f'\nBest val AUC : {study.best_value:.4f}')
print(f'Best params  :')
for k, v in study.best_params.items():
    print(f'  {k:<22} {v}')

Running Optuna search (80 trials) …


  0%|          | 0/80 [00:00<?, ?it/s]

Search finished in 6.6s

Best val AUC : 0.8060
Best params  :
  max_depth              5
  min_child_weight       18
  gamma                  3.115137743440393
  reg_alpha              0.8934880367805844
  reg_lambda             9.884053355130687
  eta                    0.06451271361515902
  subsample              0.7169676085911124
  colsample_bytree       0.850908447706292


## 4. Train Final Model with Best Hyperparameters

In [14]:
params = {
    'objective':        'binary:logistic',
    'eval_metric':      ['auc', 'aucpr'],
    'scale_pos_weight': scale_pos_weight,
    'seed':             42,
    'verbosity':        1,
    **study.best_params,
}

evals = [(dtrain, 'train'), (dval, 'val')]
evals_result = {}

print(f'Training final XGBoost with best params (top-{N_FEATURES} features)…')
t0 = time.time()
booster = xgb.train(
    params,
    dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    evals=evals,
    early_stopping_rounds=EARLY_STOPPING,
    evals_result=evals_result,
    verbose_eval=100,
)
training_time = time.time() - t0

print(f'\nTraining completed in {training_time:.2f}s')
print(f'Best iteration : {booster.best_iteration}')
print(f'Best val AUC   : {booster.best_score:.4f}')

Training final XGBoost with best params (top-50 features)…
[0]	train-auc:0.80162	train-aucpr:0.25148	val-auc:0.68324	val-aucpr:0.17789
[70]	train-auc:0.92859	train-aucpr:0.54729	val-auc:0.80595	val-aucpr:0.31457

Training completed in 0.16s
Best iteration : 40
Best val AUC   : 0.3196
